# 10.2 全书优化方法地图

一张地图收拢第 7~9 章的所有方法。**每个方法标注它的作用层次与出处**——地图本身就是复习提纲。

## 1. 优化的层次结构

```
整机耗时
 ├─ 核间（均衡性）          ← 9.2/9.3/9.4 负载均衡
 │    └─ 分核粒度: 行 → 基本块（合轴/切S2）
 ├─ 引擎间（CV 流水）        ← 7.5 / 5章 preload
 │    └─ 块大小与计算强度
 ├─ 引擎内（逐引擎）         ← 8.2~8.6
 │    ├─ Cube: K 切分
 │    ├─ Vector: VF 优化 / DN / 建模
 │    └─ Scalar: 上提/增量化/分支消除
 └─ 指令级（访存细节）       ← 7.3 / 7.6
      ├─ UB bank 冲突
      └─ RegBase / vLoop
```

**层次从粗到细，排查也从上到下**——上层的问题（某核空转 30%）在下层怎么抠都补不回来。

## 2. 方法总表

| 层次 | 方法 | 一句话 | 出处 |
|--|--|--|--|
| 建模 | Roofline | P = min(Bm·I, Pm)，先判 bound 再动手 | [7.2](../07_performance_basics/07.02_perf_modeling_roofline.ipynb) |
| 建模 | 混合带宽 | 多级存储的带宽账单，I 用混合 W/Q 算 | [7.2](../07_performance_basics/07.02_perf_modeling_roofline.ipynb) |
| 访存 | bank 冲突三板斧 | 指令限宽、读写 pingpong、多 bank group | [7.3](../07_performance_basics/07.03_memory_performance.ipynb) |
| 核间 | 有效块计量 | 分核按 (b,n,s1) 行有效 S2 块数装箱 | [9.2](../09_performance_high_level/09.02_imbalance_sources.ipynb) |
| 核间 | 不合轴贪心 | 嵌套循环 + 封核判据，六数组结果 | [9.3](../09_performance_high_level/09.03_gs1_unmerged.ipynb) |
| 核间 | 合轴贪心 | 展平 gs1 轴，粒度到基本块 | [9.4](../09_performance_high_level/09.04_gs1_merged.ipynb) |
| 核间 | 切 S2 + merge | flash decoding，(m,l,acc) 归并 | [9.5](../09_performance_high_level/09.05_s2_merge.ipynb) |
| 引擎间 | CV 四拍 | load→mm→softmax→update 重叠编排 | [7.5](../07_performance_basics/07.05_cv_pipeline.ipynb) |
| 引擎间 | preload 流水 | C2 滞后发射，两个 Cube 掩盖一个长 Vector | [5章](../05_flash_attn_non_quantized/README.md) |
| Cube | K 切分 | 容量上限即最优，L0C 满先切 M/N | [8.2](../08_performance_advanced/08.02_cube_k_split.ipynb) / [8.3](../08_performance_advanced/08.03_cube_best_k.ipynb) |
| Vector | VF 五式 | 外提/MaskReg/合并/广播/重排 | [8.4](../08_performance_advanced/08.04_vector_vf_opt.ipynb) |
| Vector | DN 选型 | 数据干净 + S2 长 → DN | [8.5](../08_performance_advanced/08.05_vector_dn_modeling.ipynb) |
| Vector | VF 建模 | 数指令 × 查表延迟 + 依赖等待 | [8.5](../08_performance_advanced/08.05_vector_dn_modeling.ipynb) |
| Scalar | 上提/增量化/分支消除/同步精简 | 不干活的指令少占周期 | [8.6](../08_performance_advanced/08.06_scalar_opt.ipynb) |
| 指令级 | RegBase/VF | 数据驻留寄存器，vLoop 按 256B 切 | [7.6](../07_performance_basics/07.06_inner_pipeline.ipynb) |

## 3. 贯穿案例：decode 场景 3.2ms 的定位地图

「实测 3.2ms，理论 1.1ms」在地图上的走法：

```
① 建模层（7.2）: decode 的 I 很小 → 带宽受限 → 优化方向是访存与搬运
② 核间层（9.2~9.5）: decode 并行度低 → 切 S2 + merge 是否已做？
③ 引擎间（7.5）: CV 是否配平？搬运是否掩盖计算？
④ 引擎内（8.2~8.6）: 带宽受限下 Cube 的 K 切分 / Vector 的 DN / Scalar 都要复查
⑤ 指令级（7.3）: bank 冲突最后一关
```

## 图示

![全书优化方法地图](images/optimization_map.svg)

## 动手实验：把地图变成可检索的索引


In [ ]:
# 动手：构建优化方法索引 —— 按「症状」检索方法
methods = [
    # (层次, 症状关键词, 方法, 出处)
    ('建模',   '不知道瓶颈在哪',       'Roofline 建模: P = min(Bm*I, Pm)',        '7.2'),
    ('建模',   '离理论性能很远',       '混合带宽账单 + 计算强度 I 推算',           '7.2'),
    ('访存',   '搬运比理论带宽慢',     'bank 冲突三板斧: 限宽/pingpong/多bank组', '7.3'),
    ('核间',   '部分核算完在等待',     '有效块计量 + 贪心分核',                   '9.2/9.3'),
    ('核间',   'decode 并行度不足',    '切 S2 + merge (flash decoding)',         '9.5'),
    ('核间',   '分核仍有锯齿',         'GS1 合轴, 粒度到基本块',                  '9.4'),
    ('引擎间', 'Cube 和 Vector 互相等','CV 四拍重叠 / preload 流水',              '7.5'),
    ('引擎间', '块大小拿不准',         '计算强度 vs L0 容量的权衡建模',           '7.5'),
    ('Cube',   'L0 装不下',            'K 切分: 取容量上限, L0C 满先切 M/N',      '8.2/8.3'),
    ('Vector', 'VF 函数慢',            '五式: 外提/MaskReg/合并/广播/重排',        '8.4'),
    ('Vector', 'transdata 开销大',      'DN 直进通路 (无 mask/pse/drop)',          '8.5'),
    ('Scalar', '标量循环拖流水',       '上提/增量化/分支消除/同步精简',            '8.6'),
    ('指令级', 'Vector 内部串行',      'RegBase/VF: vLoop 256B + 双缓冲',         '7.6'),
]

def lookup(symptom):
    print(f"症状「{symptom}」的候选处方:")
    hits = [(lv, m, src) for lv, s, m, src in methods if any(k in symptom for k in s.split('/'))]
    if not hits:
        print("  （无直接命中, 先回 7.2 做建模）")
    for lv, m, src in hits:
        print(f"  [{lv}] {m}  (第{src}章)")

for s in ['不知道瓶颈在哪', '部分核算完在等待', 'transdata 开销大', 'VF 函数慢']:
    lookup(s); print()


### 观察结论

把方法表按症状索引后，「拿到问题不知道从哪查」变成了一次字典查询。10.3 把它扩展成完整的排查决策树。

## 本节要点

- 优化分四个层次：核间 → 引擎间 → 引擎内 → 指令级，**排查从上往下**；
- 方法地图 = 15 个方法 × 层次 × 出处，是全书第三部分的复习提纲；
- 案例走法：decode 慢 → 建模判带宽受限 → 核间查切 S2 → 引擎间查 CV → 引擎内逐项复查 → 指令级查冲突。

## 小测验

1. 优化的四个层次是什么？为什么排查必须从上往下？
2. 说出「离理论性能很远」时首先该做的方法及其出处。
3. decode 场景慢，在核间层最可能缺什么方法？
4. 方法地图里 Vector 层有三个方法，分别应对什么症状？

>>> 参考答案见 [answer/10.02_answer.txt](answer/10.02_answer.txt)
